# LatentMind V6 — Colab Notebook

**Training-first pipeline**: generates planner dataset → trains MLP head → loads agent → runs tests.

Requires: T4 or L4 GPU runtime.

In [ ]:
# Core inference + RAG
!pip install 'transformers>=4.46.0' 'sentence-transformers>=3.0.0' 'accelerate>=0.27.0'
print("✓ transformers, sentence-transformers, accelerate")

# Graph + math
!pip install 'langgraph>=0.2.0' 'bitsandbytes>=0.43.0' scipy matplotlib
print("✓ langgraph, bitsandbytes, scipy, matplotlib")

# Utils
!pip install jinja2 pydantic pymysql mysql-connector-python
print("✓ jinja2, pydantic, mysql connectors")

print("\n✓ Setup complete! Ready to load agent.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys

REPO_URL = "https://github.com/Hamza09Hamza/Latent-Djezzy.git"
REPO_DIR = "/content/Latent-Djezzy"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth=1", REPO_URL, REPO_DIR], check=True)
    print(f"cloned → {REPO_DIR}")
else:
    # always pull to get the latest code
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
    # clear cached v6 modules so reimports get fresh code
    for mod in list(sys.modules.keys()):
        if "v6" in mod:
            del sys.modules[mod]
    print(f"pulled latest code, cleared module cache")

os.chdir(REPO_DIR)
print("repo ready:", REPO_DIR)

# Check if trained planner head exists
head_path = "models/planner_head.pt"
if os.path.isfile(head_path):
    import torch
    size_mb = os.path.getsize(head_path) / 1e6
    print(f"✓ Trained planner head found ({size_mb:.1f} MB) — will skip training")
else:
    print("! Trained planner head NOT found — will need to train (5 min on T4)")

In [ ]:
import shutil, os

# Adjust DRIVE_DB to where interndb.sqlite lives on YOUR Drive.
DRIVE_DB   = "/content/drive/MyDrive/interndb.sqlite"
LOCAL_DB   = "/content/interndb.sqlite"

if not os.path.isfile(LOCAL_DB):
    shutil.copy(DRIVE_DB, LOCAL_DB)
    print(f"copied {os.path.getsize(LOCAL_DB):,} bytes → {LOCAL_DB}")
else:
    print("SQLite already present:", LOCAL_DB)

In [ ]:
import os, warnings, logging

# Suppress noisy transformers deprecation warnings
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

os.environ["V6_USE_SQLITE"]    = "1"
os.environ["V6_SQLITE_PATH"]   = "/content/interndb.sqlite"
os.environ["V6_4BIT"]          = "1"          # 4-bit NF4: 3B→~3 GB, frees 3+ GB for KV cache
os.environ["V6_SPECULATIVE"]   = "0"          # off — drafter wastes VRAM on T4
os.environ["V6_SLM_OVERRIDE"]  = "Qwen/Qwen2.5-Coder-3B-Instruct"
os.environ["V6_PLANNER"]       = "prototype"  # overridden to mlp after training
os.environ["V6_OUTPUT_DIR"]    = "/content/drive/MyDrive/v6_output"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"  # reduce fragmentation

import torch
device = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
print(f"GPU: {device}  ({total_gb:.1f} GB)")
print(f"Model:              3B in 4-bit NF4 (~3 GB VRAM, quality same as f16)")
print(f"Speculative dec.:   OFF")
print(f"Output dir:         /content/drive/MyDrive/v6_output")

In [ ]:
import os

head_path = "models/planner_head.pt"
if os.path.isfile(head_path):
    print(f"Skipping dataset generation (trained head already exists)")
else:
    # Generate synthetic labelled examples for the planner MLP head.
    # Output: v6/data/planner_train.jsonl  (~1 273 examples)
    !python3 -m v6.planner_data
    !wc -l v6/data/planner_train.jsonl

In [ ]:
import os

head_path = "models/planner_head.pt"
if os.path.isfile(head_path):
    print(f"Planner head already trained; skipping training")
else:
    # Train 1024→256→(intent, caps) head — ~80 epochs, <5 min on T4.
    # Output: models/planner_head.pt
    !python3 -m v6.train_planner

os.environ["V6_PLANNER"] = "mlp"
print("Planner mode set to MLP.")

In [ ]:
import sys, os, torch
sys.path.insert(0, "/content/Latent-Djezzy")

from v6.graph import LatentMindV6
from v6.slm import get_slm

print("Loading BGE-M3 encoder + SLM (downloads on first run)…")
agent = LatentMindV6()
get_slm()  # force-load now so GPU is fully allocated before any query

free_gb  = torch.cuda.memory_reserved() / 1e9
alloc_gb = torch.cuda.memory_allocated() / 1e9
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"Agent ready — GPU: {alloc_gb:.1f} GB allocated / {total_gb:.1f} GB total")
print(f"Headroom for KV cache: ~{total_gb - alloc_gb:.1f} GB")

In [ ]:
from v6.config import V6Config

print("\n" + "="*60)
print("Configuration Verification")
print("="*60)
print(f"SLM model:            {V6Config.slm_id()}")
print(f"Speculative decoding: {'✓ Enabled' if V6Config.USE_SPECULATIVE else '✗ Disabled'}")
print(f"4-bit quantization:   {'✓ Enabled' if V6Config.USE_4BIT else '✗ Disabled'}")
print(f"Router max tokens:    {V6Config.ROUTER_MAX_NEW_TOKENS}")
print(f"SQL max tokens:       {V6Config.SQLGEN_MAX_NEW_TOKENS}")
print(f"Planner mode:         {V6Config.PLANNER_MODE}")
print("="*60 + "\n")
print("Ready to run queries. Try: ask('Hello, what can you do?')")

In [ ]:
import time
from v6.state import initial_state
from IPython.display import display, Image, Markdown

def ask(question: str, thread: str = "main") -> None:
    """Stream node-by-node progress, print the answer, display chart/email."""
    config = {"configurable": {"thread_id": thread}}
    state  = initial_state(question, thread)

    final_answer = ""
    chart_path   = ""
    email_draft  = None
    t0 = time.time()

    print(f"\n{'='*60}")
    print(f"Q: {question}")
    print(f"{'='*60}")

    for event in agent.graph.stream(state, config=config, stream_mode="updates"):
        for node_name, data in event.items():
            if not data:
                continue
            if node_name == "plan":
                ps     = data.get("plan_scores") or {}
                score  = ps.get("score", 0.0)
                intent = data.get("intent", "?")
                caps   = data.get("capabilities", [])
                print(f"  [plan]   intent={intent}  conf={score:.2f}  caps={caps}")

            elif node_name == "sql_generate":
                sql = data.get("sql", "")
                if sql:
                    short = sql.replace("\n", " ")[:80]
                    print(f"  [sql]    {short}…" if len(sql) > 80 else f"  [sql]    {short}")

            elif node_name == "sql_execute":
                rows = data.get("rows") or []
                err  = data.get("sql_error", "")
                if err:
                    print(f"  [exec]   ERROR: {err[:60]}")
                else:
                    print(f"  [exec]   {len(rows)} row(s)")

            elif node_name in ("answer", "direct_answer"):
                final_answer = data.get("final_answer", "")

            elif node_name == "visualize":
                chart_path = data.get("chart_path", "")
                if chart_path:
                    print(f"  [chart]  saved → {chart_path}")

            elif node_name == "email":
                email_draft = data.get("email_draft")
                if email_draft:
                    print(f"  [email]  drafted → {email_draft.get('to_name', '?')}")

            elif node_name == "finalize":
                if not final_answer:
                    final_answer = data.get("final_answer", "")
                if not chart_path:
                    chart_path = data.get("chart_path", "")
                if not email_draft:
                    email_draft = data.get("email_draft")

    elapsed = time.time() - t0
    print(f"\nAnswer ({elapsed:.1f}s):")
    print(final_answer or "(no answer captured)")

    # ── display chart inline ──────────────────────────────────────────────
    if chart_path and os.path.isfile(chart_path):
        print()
        display(Image(chart_path))

    # ── display email draft + save to Drive ──────────────────────────────
    if email_draft:
        print()
        to   = email_draft.get("to") or "(no recipient matched)"
        subj = email_draft.get("subject", "")
        body = email_draft.get("body", "")
        display(Markdown(f"**Email draft**\n\n"
                         f"**To:** {email_draft.get('to_name','?')} <{to}>  \n"
                         f"**Subject:** {subj}\n\n---\n\n{body}"))
        try:
            import datetime
            email_dir = "/content/drive/MyDrive/v6_output/emails"
            os.makedirs(email_dir, exist_ok=True)
            stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
            email_path = os.path.join(email_dir, f"email_{stamp}.txt")
            with open(email_path, "w") as f:
                f.write(f"To: {email_draft.get('to_name','?')} <{to}>\n")
                f.write(f"Subject: {subj}\n\n")
                f.write(body)
            print(f"  Email saved → {email_path}")
        except Exception as e:
            print(f"  (Could not save email to Drive: {e})")

    # ── free KV cache so re-running cells doesn't OOM ────────────────────
    try:
        from v6.slm import get_slm
        get_slm().clear_thread(thread)
    except Exception:
        pass
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print()

In [ ]:
ask("Hello, what can you do?")

In [ ]:
ask("What is ARPU?")

In [ ]:
ask("What is the total revenue for Oran in 2024?")

In [ ]:
ask("Show me a bar chart of total revenue by wilaya for 2024")

In [ ]:
ask("Compare churn rates between Algiers and Constantine last quarter")

In [ ]:
ask("Show me a bar chart of monthly revenue by wilaya for Q1 2024")

In [ ]:
ask("Generate an executive report for Q4 2024 performance")

In [ ]:
ask("Which wilaya had the highest churn rate?")  # follow-up — tests memory

In [ ]:
import torch
if torch.cuda.is_available():
    alloc  = torch.cuda.memory_allocated()  / 1e9
    reserv = torch.cuda.memory_reserved()   / 1e9
    total  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU  allocated: {alloc:.2f} GB")
    print(f"GPU  reserved:  {reserv:.2f} GB")
    print(f"GPU  total:     {total:.2f} GB")

In [ ]:
# Direct streaming — bypasses the graph, useful for quick SLM checks.
from v6.slm import get_slm

slm = get_slm()
messages = [
    {"role": "system", "content": "You are a helpful telecom analyst."},
    {"role": "user",   "content": "List the top 3 KPIs for a telecom operator."},
]
print("Streaming response:")
for token in slm.stream_generate(messages, max_new_tokens=256):
    print(token, end="", flush=True)
print()